# 🐱 さとりねこ 止め絵を動かす（Wan 2.2 5B・お試し）
上のメニュー **すべてのセルを実行** を押すだけ（T4は自動）。
- 止め絵を GitHub（anime-work）から取ってきて、1枚の絵から2秒の動画を作ります
- 終わると `wan_stills.zip` がダウンロードされます → デスクトップの `satorineko_anime` に移す


In [ ]:
#@title ① 準備（10分くらい）
import os, subprocess, sys, time, urllib.request, shutil
os.chdir('/content')
if not os.path.exists('ComfyUI'):
    !git clone -q https://github.com/comfyanonymous/ComfyUI
    !pip -q install -r ComfyUI/requirements.txt
    !pip -q install huggingface_hub
from huggingface_hub import hf_hub_download
M='/content/ComfyUI/models'
def get(repo, fn, sub):
    d=f'{M}/{sub}'; os.makedirs(d, exist_ok=True); dst=f'{d}/{os.path.basename(fn)}'
    if os.path.exists(dst): return
    p=hf_hub_download(repo, fn); shutil.move(os.path.realpath(p), dst)
print('モデルをダウンロード中（合計18GBほど）...')
get('Comfy-Org/Wan_2.2_ComfyUI_Repackaged','split_files/diffusion_models/wan2.2_ti2v_5B_fp16.safetensors','diffusion_models')
get('Comfy-Org/Wan_2.2_ComfyUI_Repackaged','split_files/vae/wan2.2_vae.safetensors','vae')
get('Comfy-Org/Wan_2.1_ComfyUI_repackaged','split_files/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors','text_encoders')
!rm -rf /root/.cache/huggingface
print('ComfyUIを起動中...')
log=open('/content/comfy.log','w')
srv=subprocess.Popen([sys.executable,'main.py','--listen','127.0.0.1','--port','8188','--disable-auto-launch','--lowvram'],cwd='/content/ComfyUI',stdout=log,stderr=subprocess.STDOUT)
ok=False
for _ in range(300):
    try: urllib.request.urlopen('http://127.0.0.1:8188/system_stats',timeout=2); ok=True; break
    except Exception:
        if srv.poll() is not None: break
        time.sleep(2)
if not ok:
    log.flush(); print(''.join(open('/content/comfy.log',errors='replace').readlines()[-60:]))
    raise SystemExit('起動に失敗しました。上のログをClaudeに見せてください')
print('✅ 準備完了！')


In [ ]:
#@title ② 止め絵を動かす
ステップ数 = 20  #@param {type:"integer"}
import json, glob, os, time, shutil, subprocess, urllib.request
from google.colab import files
from PIL import Image, ImageOps
STYLE='hand-drawn ink and watercolor illustration on warm paper, thick black outlines, cute round white cat with closed smiling eyes and a red scarf, static camera, gentle subtle motion, consistent style'
JOBS=[  # (名前, 画像, 動きの説明, コマ数)
 ('cut13','cut13/1.png','a clay pot of mushroom hot pot bubbling over a small fire, steam rising and swirling softly, flames flickering, the white cat sits beside it with paws together, breathing gently. ',49),
 ('cut02','stills/cut02.png','a white cat carrying a basket of mushrooms walks slowly to the right past a small wooden mushroom stall, a few red maple leaves drifting down. ',49),
 ('cut03','stills/cut03.png','a huge pile of mushrooms, the white cat peeks out from behind the mushrooms looking happy and proud, two golden sparkles twinkle above. ',49),
 ('cut05','stills/cut05.png','the white cat wearing a twisted headband holds a kitchen knife upright with determination, the blade glints, the cat breathes calmly, maple leaves sway slightly. ',49),
 ('cut08','stills/cut08.png','the white cat wearing a twisted headband holds a king oyster mushroom and tears a thin strip off it with its paw. ',49),
 ('cut12','stills/cut12.png','the white cat wearing a twisted headband raises a wooden ladle proudly beside a lidded clay pot, a little steam puffs from the pot. ',49),
 ('cut15','cut15/1.png','the white cat sits with a full round belly and closed eyes, belly slowly rising and falling, the small turtle beside it nods sleepily, red flowers and grass sway in a light breeze. ',73),
 ('cut16','cut16/1.png','sunset, seen from behind, the white cat and a small turtle sit side by side looking at the mountains, pampas grass sways in the breeze, a maple leaf drifts down, the big moon glows softly. ',73),
]
NEG='blurry, distorted, deformed, morphing, extra limbs, face change, text, watermark, 3d render, realistic photo, camera zoom, camera pan, flicker, low quality'
W='/content/src'; shutil.rmtree(W,ignore_errors=True)
subprocess.run(f'git clone -q --depth 1 -b anime-work https://github.com/masakookawa0301-byte/satori-neko-video {W}',shell=True,check=True)
BASE=f'{W}/anime_work/03_kinokonabe'
OUT='/content/wan_out'; os.makedirs(OUT,exist_ok=True)
IN='/content/ComfyUI/input'; os.makedirs(IN,exist_ok=True)
def run(name,path,prompt,コマ数):
    im=ImageOps.fit(Image.open(os.path.normpath(f'{BASE}/{path}')).convert('RGB'),(832,480),Image.LANCZOS); im.save(f'{IN}/{name}.png')
    wf={"1":{"class_type":"UNETLoader","inputs":{"unet_name":"wan2.2_ti2v_5B_fp16.safetensors","weight_dtype":"default"}},
        "2":{"class_type":"CLIPLoader","inputs":{"clip_name":"umt5_xxl_fp8_e4m3fn_scaled.safetensors","type":"wan"}},
        "3":{"class_type":"VAELoader","inputs":{"vae_name":"wan2.2_vae.safetensors"}},
        "4":{"class_type":"ModelSamplingSD3","inputs":{"model":["1",0],"shift":8.0}},
        "5":{"class_type":"CLIPTextEncode","inputs":{"text":prompt+STYLE,"clip":["2",0]}},
        "6":{"class_type":"CLIPTextEncode","inputs":{"text":NEG,"clip":["2",0]}},
        "7":{"class_type":"LoadImage","inputs":{"image":f"{name}.png"}},
        "8":{"class_type":"Wan22ImageToVideoLatent","inputs":{"vae":["3",0],"width":832,"height":480,"length":コマ数,"batch_size":1,"start_image":["7",0]}},
        "9":{"class_type":"KSampler","inputs":{"model":["4",0],"positive":["5",0],"negative":["6",0],"latent_image":["8",0],"seed":42,"steps":ステップ数,"cfg":5.0,"sampler_name":"uni_pc","scheduler":"simple","denoise":1.0}},
        "10":{"class_type":"VAEDecode","inputs":{"samples":["9",0],"vae":["3",0]}},
        "11":{"class_type":"SaveImage","inputs":{"images":["10",0],"filename_prefix":name}}}
    for f in glob.glob(f'/content/ComfyUI/output/{name}_*.png'): os.remove(f)
    pid=json.loads(urllib.request.urlopen(urllib.request.Request('http://127.0.0.1:8188/prompt',data=json.dumps({"prompt":wf}).encode(),headers={'Content-Type':'application/json'})).read())['prompt_id']
    while True:
        time.sleep(10); h=json.loads(urllib.request.urlopen(f'http://127.0.0.1:8188/history/{pid}').read())
        if pid in h:
            st=h[pid].get('status',{})
            if st.get('status_str')=='error': print('  エラー:',json.dumps(st.get('messages',[]))[-800:]); return
            break
    fd=f'{OUT}/{name}_frames'; shutil.rmtree(fd,ignore_errors=True); os.makedirs(fd)
    for f in sorted(glob.glob(f'/content/ComfyUI/output/{name}_*.png')): shutil.copy(f,fd)
    subprocess.run(['ffmpeg','-y','-loglevel','error','-framerate','24','-pattern_type','glob','-i',f'{fd}/*.png','-c:v','libx264','-pix_fmt','yuv420p',f'{OUT}/{name}.mp4'])
for name,path,prompt,n in JOBS:
    if os.path.exists(f'{OUT}/{name}.mp4'): print('済',name); continue
    t0=time.time(); print(f'▶ {name} 生成中...'); run(name,path,prompt,n); print(f'  完了 {int(time.time()-t0)}秒')
subprocess.run('cd /content/wan_out && zip -q -r /content/wan_stills.zip .',shell=True)
print('✅ 全部完了！ wan_stills.zip をダウンロードします'); files.download('/content/wan_stills.zip')
